In [ ]:
import os
import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# ---- API Key ----
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")  # <-- apni key daal
    
DB_PATH = "./chroma_db"

# ---------- 1. Load vector store ----------
print("Loading vector store...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
print(f"Vector store loaded. Total vectors: {vectorstore._collection.count()}")

# ---------- 2. Retriever ----------
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# ---------- 3. LLM ----------
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2)

# ---------- 4. Tool 1: Document search ----------
@tool
def search_documents(query: str) -> str:
    """Search the user's personal knowledge base (Excel notes, SQL notes,
    ML Notes, Power BI notes, Ethics notes, Indian Constitution) to answer
    a question. ALWAYS try this tool FIRST before web_search."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found in the personal documents."
    return "\n\n".join(d.page_content for d in docs)

# ---------- 5. Tool 2: Web search ----------
ddg_search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web for general knowledge questions that are NOT covered
    in the personal documents (e.g. general facts, current events).
    Only use this if search_documents did not return a useful answer."""
    return ddg_search.run(query)

# ---------- 6. Bind tools to LLM ----------
tools = [search_documents, web_search]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

# ---------- 7. System prompt ----------
SYSTEM_PROMPT = """You are a helpful assistant with access to two tools:
1. search_documents - searches the user's personal notes (Excel, SQL, ML, Power BI, Ethics, Indian Constitution)
2. web_search - searches the internet for general knowledge

ALWAYS try search_documents FIRST. Only use web_search if search_documents
does not contain a relevant answer, or if the question is clearly about
general/current knowledge not related to the personal documents.

After getting tool results, answer the user's question clearly and mention
which source (documents or web) the answer came from."""

# ---------- 8. Agent loop ----------
def agent_ask(question, verbose=True):
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=question)
    ]

    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)

    step = 1
    while ai_msg.tool_calls:
        for tool_call in ai_msg.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            if verbose:
                print(f"  [Step {step}] Agent chose tool: '{tool_name}' with query: {tool_args}")

            selected_tool = tool_map[tool_name]
            tool_result = selected_tool.invoke(tool_args)

            messages.append(ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_call["id"]
            ))
        step += 1
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)

    return ai_msg.content

# ---------- 9. TEST ----------
print("\n" + "="*60)
print("TEST 1: Should use DOCUMENTS")
print("="*60)
answer1 = agent_ask("What is the Status Bar in Excel used for?")
print(f"\nFINAL ANSWER: {answer1}\n")

print("="*60)
print("TEST 2: Should use WEB SEARCH (not in docs)")
print("="*60)
answer2 = agent_ask("What is the capital of Australia?")
print(f"\nFINAL ANSWER: {answer2}\n")

print("="*60)
print("TEST 3: Should use DOCUMENTS")
print("="*60)
answer3 = agent_ask("ujwal gmail?")
print(f"\nFINAL ANSWER: {answer3}\n")

Loading vector store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store loaded. Total vectors: 6183

TEST 1: Should use DOCUMENTS
  [Step 1] Agent chose tool: 'search_documents' with query: {'query': 'Status Bar Excel used for'}
  [Step 2] Agent chose tool: 'web_search' with query: {'query': 'Status Bar in Excel used for'}

FINAL ANSWER: The **Status Bar** in Excel is the horizontal bar at the bottom of the workbook window. It provides quick, real‑time information about the current worksheet and selection, such as:

| What it shows | Typical use |
|---------------|-------------|
| **Sum, Count, Average** of the selected cells | Quick calculations without inserting a formula |
| **Number of cells** in the current selection | Useful for large ranges |
| **Zoom level** | Adjusts the view of the sheet |
| **View options** (Normal, Page Layout, Page Break Preview) | Switches between worksheet views |
| **Worksheet state** (e.g., “Edit” when a cell is being edited) | Indicates the current mode |
| **Custom messages** (e.g., “Protected” or “Shared”) 